# Multi Threading in Python
Can't achieve parallelism in CPU bound task, but good for I/P bound tasks

In [1]:
# imports
import threading
import time

### Ways to create a Thread

In [15]:
# Pass a Function Directly
def worker():
    print("Thread Started")
    time.sleep(1)
    print("Thread Stopped")

t = threading.Thread(target=worker)

t.start()
t.join()    # wait for the thread to finish

Thread Started
Thread Stopped


In [21]:
# make a Thread Class

class myThread(threading.Thread):
    def run(self):
        print("Thread Started")
        time.sleep(1)
        print("Thread Stopped")

t = myThread()

t.start()
t.join()

Thread Started
Thread Stopped


In [23]:
t.start()
t.join()

RuntimeError: threads can only be started once

### Some learnings
- We can't start multiple instances of the same thread at once, in this way for now!

## Lock Object
To avoid racing conditions, as even with GIL it may happen that one thread reads the value of the shared memory before the earlier thread has completely executed, as Python instructions are not ATOMIC, so it may happen!

In [ ]:
counter = 0

def increment():
    global counter
    for _ in range(1000):
        counter += 1
        time.sleep(0.001)  # forcing thread switching to actually see the problem! (as this race condition is not visible many times)

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)
t1.start()
t2.start()
t1.join()
t2.join()

print(counter)


2022


In [172]:
# making it safe using Locks

counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(1000):
        # lock.acquire()
        # counter += 1
        # time.sleep(0.001)
        # lock.release()
        with lock:  # automatically lock.acquire() and lock.release()
            counter += 1
            time.sleep(0.001)

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)
t1.start()
t2.start()
t1.join()
t2.join()

print(counter)

2000


# RLock: Re entrant Lock
makes a thread to acquire the same lock multiple times without blocking itself, needed in
- recursive functions
- calling functions that uses the same lock inside each other

In [ ]:
# without R lock
lock = threading.Lock()

def outer():
    with lock:
        print("Outer Lock Acquired")
        inner()

def inner():
    with lock:
        print("Inner Lock Aquired")

t = threading.Thread(target=outer)
t.start()
t.join()

# will cause a deadlock as the inner one will wait for the lock to be freed but the outer will not free the lock untill inner runs!

Outer Lock Acquired


KeyboardInterrupt: 

In [179]:
# therefore use RLock
lock = threading.RLock()

def outer():
    with lock:
        print("Outer Lock Acquired")
        inner()

def inner():
    with lock:
        print("Inner Lock Aquired")

t = threading.Thread(target=outer)
t.start()
t.join()

# This will allow the same thread to acquire the lock multiple times!, thus not giving a deadlock here

Outer Lock Acquired
Inner Lock Aquired


### Why not go always with RLock?
RLock comes with it's own extra overhead, can hide logical mistakes! Therefore use RLock only when we want Re entry, otherwise use simple Locks (about ~95% time simple Lock is used)

# Semaphores
A semaphore is a synchronization object that allows only N threads to access a shared resource at a same time, i.e Semaphores are perfect when we want to limit concurrency, not completely block it like a lock! 

* Semaphores can be used as a Lock (Binary Semaphore)

In [ ]:
# example only 3 connections to the db is allowed at a same time, so we can restrict max of 5 threads to access the db at a single time

sem = threading.Semaphore(3)

def task(i):
    with sem:
        print(f"Thread {i} entered")
        time.sleep(1)
        print(f"Thread {i} left")

t1 = threading.Thread(target=task(1))
t2 = threading.Thread(target=task(2))
t3 = threading.Thread(target=task(3))
t4 = threading.Thread(target=task(4))

t1.start()
t2.start()
t3.start()
t4.start()
t1.join()
t2.join()
t3.join()
t4.join()

# GIL action can be seen as at a time only one thread is running!

Thread 1 entered
Thread 1 left
Thread 2 entered
Thread 2 left
Thread 3 entered
Thread 3 left
Thread 4 entered
Thread 4 left


# Condition Object
Used when threads needs to wait for certain condition!

In [187]:
# example: Producer Consumer Problem
condition = threading.Condition()
queue = []

def consumer():
    with condition:
        print("Consumer waiting")
        condition.wait()    # wait untill notified
        print("Consumer consumed", queue.pop())

def producer():
    time.sleep(2)
    with condition:
        queue.append("Item")
        print("Item added")
        condition.notify()  # wake up one waiting thread

t1 = threading.Thread(target=consumer)
t2 = threading.Thread(target=producer)

t1.start(), t2.start()
t1.join(), t2.join()

Consumer waiting
Item added
Consumer consumed Item


(None, None)

# Thread Communication Using Queue
using the queue.Queue() gives us the cleanest, safest, and most common way to pass data between threads in Python. It automatically solves
- thread safe
- locking
- race condition
- producer consumer sync

Because Queue is internally protected by a Lock + Condition, so we are not manually required to write Locks

### Hence this is also the industry standard to connect Threads!

In [191]:
import queue

q = queue.Queue()

def producer():
    for i in range(5):
        item = f"Task-{i}"
        print("Producing:", item)
        q.put(item)            # put with thread safety
        time.sleep(1)

def consumer():
    while True:
        item = q.get()         # waits until an item is available
        print("Consuming:", item)
        q.task_done()          # mark task as completed

p = threading.Thread(target=producer)
c = threading.Thread(target=consumer, daemon=True)

p.start()
c.start()

p.join()
q.join()


Producing: Task-0
Consuming: Task-0
Producing: Task-1
Consuming: Task-1
Producing: Task-2
Consuming: Task-2
Producing: Task-3
Consuming: Task-3
Producing: Task-4
Consuming: Task-4
